In [ ]:
import numpy as np
import scipy.linalg
import matplotlib.pyplot as plt

In [ ]:
def kinetic_energy_mels(i, j, dx):
    return np.where(
        i==j,
        np.pi ** 2 / (6 * dx ** 2),
        (-1.0) ** (i_min_j := i - j) / (dx ** 2 * i_min_j **2),
    )

In [ ]:
grid = np.linspace(-10, 10, 201)
dx = np.abs(grid[1] - grid[0])
inds=np.arange(len(grid))
potential = lambda x, omega=1: 0.5 * omega **2 * x ** 2

In [ ]:
t = kinetic_energy_mels(inds[:, None], inds[None, :], dx)

In [ ]:
h = t + np.diag(potential(grid))

In [ ]:
eps, C = scipy.linalg.eigh(h)

In [ ]:
eps[:20]

In [ ]:
def shielded_coulomb(x_1, x_2, kappa=1, a=0.01):
    return kappa / np.sqrt((x_1-x_2)**2+a**2)

In [ ]:
class HCImag:
    def __init__(self, x_min, x_max, num_dvr, potential):
        self.grid = np.linspace(x_min, x_max, num_dvr)
        self.dx=np.abs(self.grid[1]-self.grid[0])
        self.inds = np.arange(len(self.grid))
        self.potential = potential
        self.h = (
            kinetic_energy_mels(self.inds[:, None], self.inds[None, :], self.dx)
            +np.diag(self.potential(self.grid))
        )

    def __call__(self, t, c):
        C = c.reshape(self.h.shape[1], -1)
        return (-self.h @ C).ravel()

In [ ]:
class RK4:
    def __init__(self, rhs, y0, t0):
        self.rhs = rhs
        self.y = y0
        self.t = t0

    def integrate(self, t):
        dt = t - self.t

        k1 = self.rhs(self.t, self.y)
        k2 = self.rhs(self.t + dt / 2, self.y + dt * k1 / 2)
        k3 = self.rhs(self.t + dt / 2, self.y + dt * k2 / 2)
        k4 = self.rhs(self.t + dt, self.y + dt * k3)

        self.y += 1 / 6 * (k1 + 2 * k2 + 2 * k3 + k4) * dt
        self.t += dt

        return self.t, self.y

In [ ]:
t_0 = 0
c_0 = np.ones(len(inds))/np.sqrt(len(inds) * dx)
assert abs(np.dot(c_0, c_0) * dx - 1) < 1e-12

hc_imag = HCImag(grid[0], grid[-1], len(grid), potential)


In [ ]:
rk4 = RK4(hc_imag, c_0, t_0)

num_steps = 1000
dt = 1e-3
energies = np.zeros(num_steps)

for i in range(num_steps):
    energies[i] = rk4.y.T @ (hc_imag(rk4.t, rk4.y) * dx)
    rk4.integrate(rk4.t + dt)
    rk4.y = rk4.y/np.sqrt(rk4.y@rk4.y * dx)


In [ ]:
print(energies)

In [ ]:
plt.semilogy(np.abs(energies-0.5))
plt.show()

In [ ]:
class FCImag:
    def __init__(self, num_hf, num_occ, x_min, x_max, num_dvr, potential, kappa = 1):
        self.num_hf = num_hf
        self.num_occ = num_occ
        assert num_occ % 2 == 0
        self.o = slice(0, num_occ//2)
        self.grid = np.linspace(x_min, x_max, num_dvr)
        self.dx=np.abs(self.grid[1]-self.grid[0])
        self.inds = np.arange(len(self.grid))
        self.potential = potential
        self.h = (
            kinetic_energy_mels(self.inds[:, None], self.inds[None, :], self.dx)
            +np.diag(self.potential(self.grid))
        )
        self.u = shielded_coulomb(self.grid[:, None], self.grid[None, :], kappa=kappa)

    def __call__(self, t, c):
        C = c.reshape(self.h.shape[1], self.num_hf)
        D = np.einsum("pi, qi -> pq", C, C.conj())
        u_d = np.einsum("pq, q -> p", self.u, np.diag(D))
        u_ex = self.u * D
        f = self.h + 2*np.diag(u_d) - u_ex
        return (-f @ C).ravel()

In [ ]:
t_0 = 0
c_0 = np.ones(len(inds))/np.sqrt(len(inds) * dx)
assert abs(np.dot(c_0, c_0) * dx - 1) < 1e-12

fc_imag = FCImag(1, 2, grid[0], grid[-1], len(grid), potential)
rk4 = RK4(fc_imag, c_0, t_0)

num_steps = 1000
dt = 1e-3
energies = np.zeros(num_steps)

for i in range(num_steps):
    energies[i] = rk4.y.T @ (fc_imag(rk4.t, rk4.y) * dx)
    rk4.integrate(rk4.t + dt)
    rk4.y = rk4.y/np.sqrt(rk4.y@rk4.y * dx)

print(energies[-1])

plt.semilogy(np.abs(energies-0.5))
plt.show()
